# Table 1: Manufacturer and Demographics by Session

Builds a summary table from the harmonized data with:
- **Session**: Baseline, Year 2, Year 4, Year 6 (and Total)
- **Manufacturer (n)**: Counts for Siemens, GE, Philips and total per session
- **Demographics**: Age (Mean [SD]) and Sex (M/F) per session

Also reports the number of unique subjects (`subject_id`).

## Setup

In [1]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(arrow)
  library(fs)
  library(jsonlite)
})

# Config and project root
config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) {
  stop("Could not locate config.json. Set CONFIG_PATH or run from the project tree.")
}
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

harm_path <- fs::path(project_root, "data", "harmonized_data", "merged_data_meisler_analyses_harmonized.parquet")
if (!file.exists(harm_path)) stop("Harmonized parquet not found: ", harm_path)

## Load data and unique subject count

In [2]:
cols_needed <- c("subject_id", "session_id", "scanner_manufacturer", "age", "sex")
df <- arrow::read_parquet(harm_path, col_select = cols_needed) %>%
  as_tibble()

n_unique_subjects <- n_distinct(df$subject_id)
cat("Number of unique subjects (subject_id):", n_unique_subjects, "\n")

Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(cols_needed)

  # Now:
  data %>% select(all_of(cols_needed))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”


Number of unique subjects (subject_id): 10740 


## Session labels and sex display

Map `session_id` to table labels (Baseline, Year 2, Year 4, Year 6) and code sex as M/F for display.

In [3]:
# Session display order and labels (ABCD: ses-00A = baseline, ses-02A = 2-year, etc.)
session_order <- c("ses-00A", "ses-02A", "ses-04A", "ses-06A")
session_labels <- c(
  "ses-00A" = "Baseline",
  "ses-02A" = "Year 2",
  "ses-04A" = "Year 4",
  "ses-06A" = "Year 6"
)

# Sex: ABCD convention 1 = Male, 2 = Female
df <- df %>%
  mutate(
    session_label = recode(as.character(session_id), !!!session_labels, .default = as.character(session_id)),
    session_label = factor(session_label, levels = c("Baseline", "Year 2", "Year 4", "Year 6")),
    sex_display = case_when(
      sex == 1 ~ "M",
      sex == 2 ~ "F",
      TRUE ~ NA_character_
    )
  )

## Build Table 1 by session

In [4]:
# Manufacturer counts by session (one row per subject-session)
manuf_by_session <- df %>%
  filter(!is.na(scanner_manufacturer)) %>%
  count(session_id, session_label, scanner_manufacturer, name = "n") %>%
  pivot_wider(names_from = scanner_manufacturer, values_from = n, values_fill = 0L)

# Ensure Siemens, GE, Philips columns exist
for (col in c("Siemens", "GE", "Philips")) {
  if (!col %in% names(manuf_by_session)) manuf_by_session[[col]] <- 0L
}
manuf_by_session <- manuf_by_session %>%
  mutate(Total_manuf = Siemens + GE + Philips)

# Demographics by session: age mean (SD), sex M/F
demo_by_session <- df %>%
  group_by(session_id, session_label) %>%
  summarise(
    age_mean = mean(age, na.rm = TRUE),
    age_sd = sd(age, na.rm = TRUE),
    n_m = sum(sex_display == "M", na.rm = TRUE),
    n_f = sum(sex_display == "F", na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    Age_Mean_SD = sprintf("%.2f [%.2f]", age_mean, age_sd),
    Sex_M_F = paste0(n_m, "/", n_f)
  )

# Join and keep one row per session (ordered)
sessions_ordered <- session_order[session_order %in% df$session_id]
table_by_session <- manuf_by_session %>%
  filter(session_id %in% sessions_ordered) %>%
  mutate(session_id = factor(session_id, levels = sessions_ordered)) %>%
  arrange(session_id) %>%
  left_join(
    demo_by_session %>% select(session_id, Age_Mean_SD, Sex_M_F),
    by = "session_id"
  ) %>%
  select(Session = session_label, Siemens, GE, Philips, Total_manuf, Age_Mean_SD, Sex_M_F)

## Add Total row and display

In [5]:
# Total row (all sessions)
total_manuf <- df %>%
  filter(!is.na(scanner_manufacturer)) %>%
  count(scanner_manufacturer, name = "n") %>%
  pivot_wider(names_from = scanner_manufacturer, values_from = n, values_fill = 0L)
for (col in c("Siemens", "GE", "Philips")) {
  if (!col %in% names(total_manuf)) total_manuf[[col]] <- 0L
}
total_manuf <- total_manuf %>% mutate(Total_manuf = Siemens + GE + Philips)

total_demo <- df %>%
  summarise(
    age_mean = mean(age, na.rm = TRUE),
    age_sd = sd(age, na.rm = TRUE),
    n_m = sum(sex_display == "M", na.rm = TRUE),
    n_f = sum(sex_display == "F", na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    Age_Mean_SD = sprintf("%.2f [%.2f]", age_mean, age_sd),
    Sex_M_F = paste0(n_m, "/", n_f)
  )

total_row <- tibble(
  Session = "Total",
  Siemens = total_manuf$Siemens,
  GE = total_manuf$GE,
  Philips = total_manuf$Philips,
  Total_manuf = total_manuf$Total_manuf,
  Age_Mean_SD = total_demo$Age_Mean_SD,
  Sex_M_F = total_demo$Sex_M_F
)

table1 <- bind_rows(table_by_session, total_row)
table1

Session,Siemens,GE,Philips,Total_manuf,Age_Mean_SD,Sex_M_F
<chr>,<int>,<int>,<int>,<int>,<chr>,<chr>
Baseline,5416,2174,814,8404,9.97 [0.63],4366/4038
Year 2,4336,1736,637,6709,11.99 [0.65],3604/3105
Year 4,3689,1446,516,5651,14.16 [0.72],3027/2624
Year 6,2069,1043,311,3423,16.08 [0.66],1797/1626
Total,15510,6399,2278,24187,12.37 [2.28],12794/11393


## Optional: Formatted table (e.g. for manuscript)

Uncomment and run if you have `knitr` / `kableExtra` to get a publication-ready table.

In [6]:
# Rename columns to match typical Table 1 headers (Manufacturer section: Siemens, GE, Philips, Total; Demographics: Age, Sex)
table1_display <- table1 %>%
  rename(
    Total = Total_manuf,
    `Age (Mean [SD])` = Age_Mean_SD,
    `Sex (M/F)` = Sex_M_F
  )

# Optional: kable for markdown/HTML/LaTeX
# if (requireNamespace("knitr", quietly = TRUE)) {
#   knitr::kable(table1_display, align = c("l", rep("r", 6)))
# }

print(table1_display)

# A tibble: 5 × 7
  Session  Siemens    GE Philips Total `Age (Mean [SD])` `Sex (M/F)`
  <chr>      <int> <int>   <int> <int> <chr>             <chr>      
1 Baseline    5416  2174     814  8404 9.97 [0.63]       4366/4038  
2 Year 2      4336  1736     637  6709 11.99 [0.65]      3604/3105  
3 Year 4      3689  1446     516  5651 14.16 [0.72]      3027/2624  
4 Year 6      2069  1043     311  3423 16.08 [0.66]      1797/1626  
5 Total      15510  6399    2278 24187 12.37 [2.28]      12794/11393
